In [ ]:
import base64
import json
import re
import shutil
from pathlib import Path

import requests

# -----------------------------
# LM Studio config
# -----------------------------

API_URL = "http://localhost:1234/v1/chat/completions"
MODEL_NAME = "qwen/qwen3-vl-4b"
TIMEOUT_SECONDS = 90

# -----------------------------
# Carpetas
# -----------------------------

INPUT_FOLDER = "Output"
OUTPUT_FOLDER = "Dataset/PlatesReviewed"
CORRECT_FOLDER = Path(OUTPUT_FOLDER, "correctas")
INCORRECT_FOLDER = Path(OUTPUT_FOLDER, "incorrectas")

CORRECT_FOLDER.mkdir(parents=True, exist_ok=True)
INCORRECT_FOLDER.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Prompt sistema
# -----------------------------

SYSTEM_PROMPT = """
You are a strict OCR and visual QA assistant for vehicle license plates.

TASK CONTEXT:
- You receive one image and an expected plate text extracted from the filename.
- Filename pattern example: PTC-836_00001_.png
- Only the text before the first underscore is the expected plate (PTC-836).
- Anything after the first underscore is irrelevant metadata.

YOUR GOALS:
1) Read the visible plate text from the image.
2) Verify if the image is usable for a plate dataset.
3) Detect if there are duplicated identical plates in the same image (for example two identical plates visible on screen).
4) Decide if expected text matches the visible main plate text.

USABILITY RULES:
Mark usable = false if any of these apply:
- Plate text cannot be read with confidence.
- Plate is too blurry, too occluded, too small, or overexposed.
- There are two or more identical plates visible in the same image.
- No plate is visible.

Mark usable = true only when:
- A single primary plate can be read clearly.
- There is no duplicated identical plate instance in frame.

MATCHING RULE:
- Compare expected_plate and detected_plate exactly after uppercase normalization.

OUTPUT REQUIREMENTS:
- Return ONLY valid JSON.
- Do NOT add markdown, comments, or extra text.
- Use this exact schema and keys:
{
  "expected_plate": "STRING",
  "detected_plate": "STRING",
  "is_match": true,
  "has_duplicate_identical_plate": false,
  "usable": true,
  "reason": "SHORT STRING"
}

GUIDELINES FOR detected_plate:
- Use uppercase letters/numbers/hyphen only when possible.
- If unreadable, return empty string.
"""

# -----------------------------
# Utilidades
# -----------------------------

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def extract_expected_plate(filename):
    # Toma todo antes del primer underscore.
    return filename.split("_", 1)[0].upper().strip()


def split_name_prefix_and_suffix(filename):
    if "_" in filename:
        prefix, rest = filename.split("_", 1)
        return prefix, "_" + rest
    stem = Path(filename).stem
    suffix = Path(filename).suffix
    return stem, suffix


def normalize_plate(text):
    text = (text or "").upper().strip()
    return re.sub(r"[^A-Z0-9-]", "", text)


def is_one_char_typo(expected_plate, detected_plate):
    if len(expected_plate) != len(detected_plate):
        return False
    diffs = sum(1 for a, b in zip(expected_plate, detected_plate) if a != b)
    return diffs == 1


def parse_llm_json(content):
    content = content.strip()

    # Soporta casos donde el modelo envía JSON entre markdown fences.
    if content.startswith("```"):
        content = re.sub(r"^```(?:json)?", "", content).strip()
        content = re.sub(r"```$", "", content).strip()

    data = json.loads(content)

    # Campos por defecto si faltan.
    return {
        "expected_plate": str(data.get("expected_plate", "")),
        "detected_plate": str(data.get("detected_plate", "")),
        "is_match": bool(data.get("is_match", False)),
        "has_duplicate_identical_plate": bool(
            data.get("has_duplicate_identical_plate", False)
        ),
        "usable": bool(data.get("usable", False)),
        "reason": str(data.get("reason", "")),
    }


# -----------------------------
# LLM clasificación
# -----------------------------

def classify_plate_image(image_path):
    expected_plate = extract_expected_plate(image_path.name)
    img_base64 = encode_image(image_path)

    user_prompt = (
        "Analyze this image and return strict JSON. "
        f"expected_plate={expected_plate}"
    )

    payload = {
        "model": MODEL_NAME,
        "temperature": 0,
        "max_tokens": 200,
        "stop": ["\n\n\n"],
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{img_base64}"},
                    },
                ],
            },
        ],
    }

    response = requests.post(API_URL, json=payload, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()
    result = response.json()

    content = result["choices"][0]["message"]["content"]
    llm = parse_llm_json(content)

    llm["expected_plate"] = normalize_plate(expected_plate)
    llm["detected_plate"] = normalize_plate(llm.get("detected_plate", ""))

    # Recalcula match en local para evitar inconsistencias del modelo.
    llm["is_match"] = llm["expected_plate"] == llm["detected_plate"]

    return llm

# -----------------------------
# Enrutamiento de archivos
# -----------------------------

def build_corrected_name(original_name, corrected_plate):
    _, suffix_part = split_name_prefix_and_suffix(original_name)
    return f"{corrected_plate}{suffix_part}"


def route_image(image_path, analysis):
    expected_plate = analysis["expected_plate"]
    detected_plate = analysis["detected_plate"]
    usable = analysis["usable"]
    has_duplicate = analysis["has_duplicate_identical_plate"]

    # Caso 1: inutilizable o placa duplicada => incorrectas
    if (not usable) or has_duplicate:
        destination = INCORRECT_FOLDER / image_path.name
        shutil.copy2(image_path, destination)
        return "incorrectas", destination.name, "inutilizable_o_duplicada"

    # Caso 2: coincide exacto => correctas con nombre original
    if expected_plate == detected_plate:
        destination = CORRECT_FOLDER / image_path.name
        shutil.copy2(image_path, destination)
        return "correctas", destination.name, "match_exacto"

    # Caso 3: solo 1 caracter incorrecto en nombre => renombrar a detectada y enviar a correctas
    if detected_plate and is_one_char_typo(expected_plate, detected_plate):
        corrected_name = build_corrected_name(image_path.name, detected_plate)
        destination = CORRECT_FOLDER / corrected_name
        shutil.copy2(image_path, destination)
        return "correctas", destination.name, "renombrada_por_1_caracter"

    # Caso 4: cualquier otro mismatch => incorrectas
    destination = INCORRECT_FOLDER / image_path.name
    shutil.copy2(image_path, destination)
    return "incorrectas", destination.name, "mismatch_no_corregible"


# -----------------------------
# Procesar carpeta
# -----------------------------

def classify_folder(folder):
    image_exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
    images = [
        p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in image_exts
    ]

    if not images:
        print(f"No se encontraron imágenes en: {folder}")
        return

    total = len(images)
    correct_count = 0
    incorrect_count = 0

    for i, img in enumerate(images, start=1):
        try:
            analysis = classify_plate_image(img)
            bucket, out_name, decision = route_image(img, analysis)

            if bucket == "correctas":
                correct_count += 1
            else:
                incorrect_count += 1

            print(
                f"[{i}/{total}] {img.name} -> {bucket} | {decision} | "
                f"esperada={analysis['expected_plate']} detectada={analysis['detected_plate']} "
                f"dup={analysis['has_duplicate_identical_plate']} usable={analysis['usable']} -> {out_name}"
            )

        except Exception as e:
            incorrect_count += 1
            fallback = INCORRECT_FOLDER / img.name
            try:
                shutil.copy2(img, fallback)
            except Exception:
                pass
            print(f"[{i}/{total}] Error procesando {img.name}: {e}")

    print("\nResumen final")
    print(f"Total: {total}")
    print(f"Correctas: {correct_count}")
    print(f"Incorrectas: {incorrect_count}")


# Ejecutar
classify_folder(INPUT_FOLDER)


[1/500] WO-24-UQ_00001_.png -> correctas | match_exacto | esperada=WO-24-UQ detectada=WO-24-UQ dup=False usable=True -> WO-24-UQ_00001_.png
[2/500] IYF-509_00001_.png -> correctas | match_exacto | esperada=IYF-509 detectada=IYF-509 dup=False usable=True -> IYF-509_00001_.png
[3/500] 883-XBN_00001_.png -> correctas | match_exacto | esperada=883-XBN detectada=883-XBN dup=False usable=True -> 883-XBN_00001_.png
[4/500] ACL-123_00001_.png -> correctas | match_exacto | esperada=ACL-123 detectada=ACL-123 dup=False usable=True -> ACL-123_00001_.png
[5/500] 3651-WMF_00001_.png -> correctas | match_exacto | esperada=3651-WMF detectada=3651-WMF dup=False usable=True -> 3651-WMF_00001_.png
[6/500] 794-STI_00001_.png -> correctas | match_exacto | esperada=794-STI detectada=794-STI dup=False usable=True -> 794-STI_00001_.png
[7/500] 172-IMJ_00001_.png -> correctas | match_exacto | esperada=172-IMJ detectada=172-IMJ dup=False usable=True -> 172-IMJ_00001_.png
[8/500] EB-29-SS_00001_.png -> correctas

In [6]:
def renombrar_imagenes_quitando_sufijo(directorio=INPUT_FOLDER):
    image_exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
    directorio = Path(directorio)

    renombradas = 0
    omitidas = 0

    for ruta in directorio.iterdir():
        if not ruta.is_file() or ruta.suffix.lower() not in image_exts:
            continue

        if "_" not in ruta.stem:
            continue

        nuevo_stem = ruta.stem.split("_", 1)[0]
        nuevo_nombre = f"{nuevo_stem}{ruta.suffix}"
        destino = ruta.with_name(nuevo_nombre)

        if destino.exists():
            print(f"Omitida (ya existe): {ruta.name} -> {destino.name}")
            omitidas += 1
            continue

        ruta.rename(destino)
        print(f"Renombrada: {ruta.name} -> {destino.name}")
        renombradas += 1

    print(f"\nTotal renombradas: {renombradas}")
    print(f"Total omitidas: {omitidas}")


# Ejecutar sobre INPUT_FOLDER
renombrar_imagenes_quitando_sufijo(directorio="Dataset/PlatesReviewed/correctas")

Renombrada: WO-24-UQ_00001_.png -> WO-24-UQ.png
Renombrada: IYF-509_00001_.png -> IYF-509.png
Renombrada: 883-XBN_00001_.png -> 883-XBN.png
Renombrada: 3651-WMF_00001_.png -> 3651-WMF.png
Renombrada: 794-STI_00001_.png -> 794-STI.png
Renombrada: 172-IMJ_00001_.png -> 172-IMJ.png
Renombrada: EB-29-SS_00001_.png -> EB-29-SS.png
Renombrada: 366-WUT_00001_.png -> 366-WUT.png
Renombrada: G-79509_00001_.png -> G-79509.png
Renombrada: JO-11-DI_00001_.png -> JO-11-DI.png
Renombrada: 153-PRI_00001_.png -> 153-PRI.png
Renombrada: 7655-NKY_00001_.png -> 7655-NKY.png
Renombrada: 0837-GFR_00001_.png -> 0837-GFR.png
Renombrada: RKK-554-A_00001_.png -> RKK-554-A.png
Renombrada: DX-66-LB_00001_.png -> DX-66-LB.png
Renombrada: 408-JYD_00001_.png -> 408-JYD.png
Renombrada: PFM-026_00001_.png -> PFM-026.png
Renombrada: NRD-423-0_00001_.png -> NRD-423-0.png
Renombrada: FKK-461_00001_.png -> FKK-461.png
Renombrada: S-69040_00001_.png -> S-69040.png
Renombrada: M-44207_00001_.png -> M-44207.png
Renombrada: 